# 4.4 Function Decorator

**Prerequisites:** 4.1 Functions User-defined, 4.2 Functions Builtins  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- First-class functions: assigning, passing and returning them
- Nested functions and **closures**, with `nonlocal`
- What a decorator is, and the `@` syntax it replaces
- General-purpose decorators with `*args, **kwargs`
- **Stacking order** — applied bottom-up, executed outside-in
- Decorator **factories** — decorators that take arguments
- 🔴 **`functools.wraps`** — and what breaks without it
- Class-based decorators, and decorating methods
- `functools.singledispatch`, and the decorators you already use

---

## First Class functions in Python:
- A programming language is said to support **first-class functions** if it treats functions as **first-class objects.**
<br/><br/>
- **First class objects** in a language are handled uniformly throughout. 
- In Python, functions are the first class objects, i.e, they have all the rights as other variables in the programming language have.
    - which means that it can be dynamically created, destroyed, stored in data structures, passed as arguments, passed to a function, returned as a value, or used in control structures 
<br/><br/>
- **Properties of first class functions:**
    - A function is an instance of the object type.
    - You can store the function in a variable.
    - You can pass the function as a parameter to another function.
    - You can return the function from a function.
    - You can store them in data structures such as hash tables, lists..

### Object type

In [ ]:
a=10
print(isinstance(a,object))
print(isinstance(a,int))

In [ ]:
def myfunc():
    pass

print(isinstance(myfunc,object))

In [ ]:
class var:
    pass
print(isinstance(var,object))
print(isinstance(var,type))

### Normal function

In [ ]:
def square(n):
    print(f'Square of {n} is', n*n)

print(square)
print(type(square))
print(isinstance(square,object))
square(5)

### Assigning Functions to Variable:
- Python functions are first class objects i.e, we can assign them in variable.
- Let's assign function `square` to a variable name `func`. 
    - This assignment doesn't call the function.
- It takes the function **object referenced** by `square` and creates a second name pointing to it, `func`.
    - i.e, now `func` can also be used same as `square()` by calling it.

In [ ]:
def square(n):
    print(f'Square of {n} is', n*n)

func = square # assigning function to a variable
print(square)
print(func)
square(5)
func(5)

In [ ]:
# Assigning functions to a list
def cube(n):
    print(f'Cube of {n} is', n*n*n)

sq_cube=[square,cube]
print(sq_cube[0])
print(sq_cube[1])
sq_cube[0](5)
sq_cube[1](5)

### Passing Functions as Arguments to other Functions:
- Because functions are objects we can pass them as arguments to other functions aswell. 
- Functions that can accept other functions as arguments are also called **higher-order functions**. 
    - Eg. sort(), map(), reduce(), filter() etc.

In [ ]:
def square(n):
    print(f'Square of {n} is', n*n)
    
def cube(n):
    print(f'Cube of {n} is', n*n*n)
    
def choice(func,n): #higher order function
    return func(n)

choice(square,5)
choice(cube,5)

### Nested functions in Python:
- A function which is defined inside another function is known as nested function. 
- Python allows a nested function to access the outer scope of the enclosing function.
    - i.e, Nested functions can access variables of the enclosing scope.
- In Python, these non-local variables can be accessed only within their scope and not outside their scope.

In [ ]:
def encloseFunc(): # Enclosing(Parent) Function
    print("Now we are in encloseFunc.")
    def nestedFunc(): # Nested(Inner) Function
        print("Welcome To nested")
    print("Calling the nestedFunc.")
    return nestedFunc()

print("We are in _main_.\nCalling the encloseFunc.")
encloseFunc()

In [ ]:
# nested function accessing variable of enclosing scope
def encloseFunc(n):
    def square():
        print(f'Square of {n} is', n*n) #accessing variables of the enclosing scope(non-local variable n)
    return square()

encloseFunc(5)

In [ ]:
# square() # Give an Error

#### NOTE: 
- square() is locally scoped to encloseFunc() 
    - i.e, square() exist only inside the encloseFunc() as local variables.
- The inner functions are not defined until the parent function is called.
    - Whenever we call parent function, the inner functions are also called.
    - But because of their local scope, it is not available outside of the parent function.

### Function as a return value: Python closure
- Python also allows us to use functions as return values.


### Python Closures:
- A Closure is a function object that remembers values in enclosing scopes even if they are not present in memory 
    - i.e, a closure is a way of keeping alive a variable even when the function has returned.
- A closure—unlike a plain function—allows the function to access those captured variables through the closure’s copies of their values or references, even when the function is invoked outside their scope.
<br/><br/>
- Closures is a record that stores a function together with an environment: a mapping associating each free variable of the function (variables that are used locally, but defined in an enclosing scope) with the value or reference to which the name was bound when the closure was created.
- So in a closure, a function is defined along the environment. In python this is done by nesting a function inside the encapsulating function and then returning the underlying function.

#### Python closure structure:

In [ ]:
def closureFunc():
    def nestedFunc():
        print("Welcome To Closure")
    return nestedFunc

get = closureFunc()
get() # get variable working as nestedFunc()

#### Python closure embeds data with code :

In [ ]:
def closureFunc(n):
    def square():
        print("Welcome To Closure")
        print(f'Square of {n} is', n*n)
    return square

get = closureFunc(5)
get()

#### Python Closure remembers its context:

In [ ]:
def closureFunc(n):
    def square():
        print("Welcome To Closure")
        print(f'Square of {n} is', n*n)
    return square

get = closureFunc(5)
del closureFunc
get()

#### Modifying of nonlocal variable in closures

In [ ]:
def closureFunc(n):
    def squareOneLess():
        nonlocal n
        print("Welcome To Closure")
        n-=1
        print(f'Square of {n} is', n*n)
    return squareOneLess

get = closureFunc(5)
del closureFunc
get()

- Notice that we have taken a variable `n` in the closureFunc and reuse it in the squareOneLess declaring as a `nonlocal` to this function using the keyword `nonlocal`.
- If you do not declare as nonlocal then you will get error that local variable `n referenced before assignment`, that means it will be considered as a local variable to the squareOneLess().

- **Summary:**
    - closureFunc() was called with the values 5 and it returns a function that was bound to the name `get`. 
    - On calling get(), the values were still remembered although we had already finished executing the closureFunc().
    - This technique by which some data (5) gets attached to the code is called closure in Python.
        - This value in the enclosing scope is remembered even when the variable goes out of scope or the function itself is removed from the current namespace.

#### Closure with argument
- Let's provide argument to the nestedFunc.

In [ ]:
def closureFunc(n):
    def squareXLess(x):
        nonlocal n
        print("Welcome To Closure")
        n-=x
        print(f'Square of {n} is', n*n)
    return squareXLess

get = closureFunc(5)
del closureFunc
get(2)

- All function objects have a `__closure__` attribute that returns a tuple of cell objects if it is a closure function.

In [ ]:
get.__closure__

#### Why to use Closures:
- 1) As closures are used as callback functions, they provide some sort of data hiding. This helps us to reduce the use of global variables.
- 2) When we have few functions in our code, closures prove to be efficient way. But if we need to have many functions, then go for class (OOP).


#### When do we have closure:
- The criteria that must be met to create closure in Python are summarized in the following points:
    - We must have a nested function (function inside a function).
    - The nested function must refer to a value defined in the enclosing function.
    - The enclosing function must return the nested function.

In [ ]:
def greeting(name, gender):
    def salutation():
        if gender=='M':
            return f'Mr. {name}'
        elif gender=='F':
            return f'Ms. {name}'
        else:
            return name
    return salutation

person1 = greeting('Aditya','M')
print(person1())

person2 = greeting('Neha','F')
print(person2())

- As observed from above code, closures help to invoke function outside their scope.
- The function innerFunc has its scope only inside the outerFunc. But with the use of closures we can easily extend its scope to invoke a function outside its scope.

## What is a Decorator?
- By definition, a decorator is 
    - a function that takes in another function, 
    - extends the behavior of the latter function and return it,
    - without explicitly modifying that function.
- Decorator helps to add some additional functionalities to an already defined function. 
- Decorators is also called **meta programming** as a part of the program tries to modify another part of the program at compile time.
- Decorators provide a simple syntax for calling higher-order functions.
<br/><br/> 
- A decorator is a callable that returns a callable i.e, a decorator takes in a function, adds some functionality and returns it.
    - Any object which implements the special method `__call__()` is termed callable.
- Ref: <https://realpython.com/primer-on-python-decorators/>

In [ ]:
# Want to add new feature to ordinary() is called i.e, decorating ordinary()
def decorate(function):
    def additional():
        print("Existing code got decorated")
        function()
    return additional

def existing():
    print("Existing code")

In [ ]:
existing()

In [ ]:
existing= decorate(existing)
existing()

- This is a common construct and for this reason, Python has a syntax to simplify this sometimes called the “pie” syntax. 
- We can use the @ symbol along with the name of the decorator function and place it above the definition of the function to be decorated.
```python
@decorate
def existing():
    print("Existing code")
```
- **Is equivalent to**
```python
def existing():
    print("Existing code")
existing= decorate(existing)
```

In [ ]:
def decorate(function):
    def additional():
        print("Existing code got decorated")
        function()
    return additional

@decorate
def existing():
    print("Existing code")

In [ ]:
existing()

### Applying Multiple Decorators to a Single Function:

In [ ]:
def new_decorate(function):
    def new_addition():
        print("Existing code got decorated again..")
        function()
    return new_addition

def decorate(function):
    def additional():
        print("Existing code got decorated")
        function()
    return additional

@new_decorate
@decorate
def existing():
    print("Existing code")

In [ ]:
existing()

- **NOTE: Application of decorators is from the bottom up approach.**

### Stacking order: bottom-up application, outside-in execution

The note above says decorators apply bottom-up. It is worth making that concrete, because
**two different orderings are involved** and they run opposite ways:

```
@outer
@inner
def task(): ...
```

is exactly `task = outer(inner(task))`.

| | Order |
|---|---|
| **Applied** (wrapping happens) | Bottom-up — `inner` wraps `task` first |
| **Executed** (at call time) | Top-down — `outer`'s code runs first, then `inner`'s, then the body |

**Analogy:** parcels. The last wrapper applied is the outermost layer, so it's the first one
you unwrap.

In [ ]:
from functools import wraps

def outer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("  outer: before")
        result = func(*args, **kwargs)
        print("  outer: after")
        return result
    return wrapper


def inner(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("  inner: before")
        result = func(*args, **kwargs)
        print("  inner: after")
        return result
    return wrapper


@outer
@inner
def task():
    print("  >>> task body <<<")

task()

print("""
Reading order:
    @outer          applied LAST   -> outermost -> its 'before' runs FIRST
    @inner          applied FIRST  -> innermost -> closest to the body
    def task

Equivalent to:  task = outer(inner(task))
""")

# Swap them and the output order swaps
@inner
@outer
def task2():
    print("  >>> task2 body <<<")

task2()

### Accepting Arguments in Decorator Functions:

In [ ]:
def check_division(function):
    def check(a,b):
        print(f'You are going to divide {a} by {b}')
        if b == 0:
            return "Zero Divivion not allowed!!"
        return function(a,b)
    return check

@check_division
def divide(a,b):
    return a/b

In [ ]:
num1= float(input("Enter 1st: "))
num2= float(input("Enter 2nd: "))
result= divide(num1,num2)
print(result)

### WA Decorator to Standardize Mobile Number
- You are given  mobile numbers. Sort them in ascending order then print them in the standard format shown below: +91 xxxxx xxxxx
- The given mobile numbers may have +91 ,91 or 0 written before the actual 10 digit number. Alternatively, there may not be any prefix at all.
<br/><br/>
- **Input Format:** 
    - The first line of input contains an integer N, the number of mobile phone numbers.
    - N lines follow each containing a mobile number.
    - Eg.
        - 3
        - 07895462130
        - 919875641230
        - 9195969878
<br/><br/>
- **Output Format:**
    - Print  mobile numbers on separate lines in the required format.
    - Eg.
        - +91 78954 62130
        - +91 91959 69878
        - +91 98756 41230

In [ ]:
def format_no(func):
    def wrapper(nos):
        final=[]
        for j in nos:
            final.append(f'+91 {j[-10:-5]} {j[-5:]}')
        func(final)    
    return wrapper

@format_no
def sort_phone(no_arr):
    print(*sorted(no_arr), sep='\n')

In [ ]:
arr = [input() for i in range(int(input()))]
sort_phone(arr)

### Decorate Name Directory
- Let's use decorators to build a name directory! You are given some information about N people. 
    - Each person has a first name, last name, age and sex.
    - Print their names in a specific format sorted by their age in ascending order i.e. the youngest person's name should be printed first. For two people of the same age, print them in the order of their input.
<br/><br/>
- **Input Format:**
    - The first line contains the integer N, the number of people.
    - N lines follow each containing the space separated values of the first name, last name, age and sex, respectively.
    - Eg.
        - 3
        - Mike Thomson 20 M
        - Robert Bustle 32 M
        - Andria Bustle 30 F
<br/><br/>
- **Constraints:**
    - 1<=N<=10
<br/><br/>
- **Output Format:**
    - Output N names on separate lines in the format described above in ascending order of age.
    - Eg.
        - Mr. Mike Thomson
        - Ms. Andria Bustle
        - Mr. Robert Bustl

In [ ]:
import operator
def person_lister(f):
    def inner(people):
        pass
    return inner

@person_lister
def name_format(person):
    return ("Mr. " if person[3]=="M" else "Ms.") + person[0] + " " + person[1]

if __name__ == '__main__':
    people = [input().split() for i in range(int(input()))]
    print(*name_format(people), sep='\n')

### Defining General Purpose Decorators:

In [ ]:
def deco(function):
    def wrapper(*args,**kwargs):
        print('The positional arguments are', args)
        print('The keyword arguments are', kwargs)
        function(*args)
    return wrapper

@deco
def function_with_no_argument():
    print("No arguments here.")

function_with_no_argument()

- Use the decorator using positional arguments.

In [ ]:
@deco
def function_with_arguments(a, b, c):
    print(a, b, c)

function_with_arguments(1,2,3)

- Keyword arguments are passed using keywords.

In [ ]:
@deco
def function_with_keyword_arguments():
    print("This has shown keyword arguments")

function_with_keyword_arguments(first_name="Derrick", last_name="Mwiti")

#### Passing Arguments to the Decorator:

In [ ]:
def decorator_maker_with_arguments(decorator_arg1, decorator_arg2, decorator_arg3):
    def decorator(func):
        def wrapper(function_arg1, function_arg2, function_arg3) :
            "This is the wrapper function"
            print("The wrapper can access all the variables\n"
                  "\t- from the decorator maker: {0} {1} {2}\n"
                  "\t- from the function call: {3} {4} {5}\n"
                  "and pass them to the decorated function"
                  .format(decorator_arg1, decorator_arg2,decorator_arg3,
                          function_arg1, function_arg2,function_arg3))
            return func(function_arg1, function_arg2,function_arg3)

        return wrapper

    return decorator

pandas = "Pandas"
@decorator_maker_with_arguments(pandas, "Numpy","Scikit-learn")
def decorated_function_with_arguments(function_arg1, function_arg2,function_arg3):
    print("This is the decorated function and it only knows about its arguments: {0}"
           " {1}" " {2}".format(function_arg1, function_arg2,function_arg3))

decorated_function_with_arguments(pandas, "Science", "Tools")

### 🔴 Debugging Decorators — and the fix the original notes were missing

Here is the problem the cells below reveal: **a decorated function loses its identity.**

`existing = decorate(existing)` rebinds the name to the *wrapper*. So `__name__` is now
`"wrapper"`, `__doc__` is the wrapper's docstring (usually `None`), and the signature you
see in `help()` is `(*args, **kwargs)`.

That is not cosmetic:

- `help()` and IDE tooltips show the wrong thing
- Debuggers and tracebacks name the wrong function
- **Frameworks break.** Flask routes by `__name__`; pytest collects by name; Sphinx documents
  the wrapper instead of your function
- Two different decorated functions can end up with the *same* `__name__`, which is how you
  get Flask's `View function mapping is overwriting an existing endpoint` error

### The fix: `functools.wraps`

```python
from functools import wraps

def decorate(func):
    @wraps(func)              # <- copies __name__, __doc__, __module__,
    def wrapper(*args, **kw): #    __qualname__, __dict__ and __wrapped__
        return func(*args, **kw)
    return wrapper
```

`@wraps` is itself a decorator (applied to your wrapper) built on
`functools.update_wrapper`. **There is no situation where you want to leave it out.**

In [ ]:
from functools import wraps


# ---- WITHOUT wraps: the function loses its identity ----
def broken_decorator(func):
    def wrapper(*args, **kwargs):
        """I am the wrapper's docstring."""
        return func(*args, **kwargs)
    return wrapper


@broken_decorator
def calculate_tax(amount: float, rate: float = 0.18) -> float:
    """Return the tax payable on `amount`."""
    return amount * rate


print("WITHOUT @wraps")
print("  __name__       :", calculate_tax.__name__)
print("  __doc__        :", calculate_tax.__doc__)
print("  __annotations__:", calculate_tax.__annotations__)

import inspect
print("  signature      :", inspect.signature(calculate_tax))


# ---- WITH wraps: identity preserved ----
def good_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@good_decorator
def calculate_tax(amount: float, rate: float = 0.18) -> float:
    """Return the tax payable on `amount`."""
    return amount * rate


print("\nWITH @wraps")
print("  __name__       :", calculate_tax.__name__)
print("  __doc__        :", calculate_tax.__doc__)
print("  __annotations__:", calculate_tax.__annotations__)
print("  signature      :", inspect.signature(calculate_tax))

# @wraps also leaves a way back to the original
print("\n  __wrapped__    :", calculate_tax.__wrapped__.__name__)
print("  undecorated call:", calculate_tax.__wrapped__(1000))


# ---- Why it matters in practice ----
registry = {}

def register(func):
    registry[func.__name__] = func       # keys on __name__
    return func

@register
@broken_decorator
def handler_one(): return 1

@register
@broken_decorator
def handler_two(): return 2

print("\nregistry without wraps:", list(registry),
      "<- both registered as 'wrapper', one silently overwrote the other")

In [ ]:
decorated_function_with_arguments.__name__

In [ ]:
decorated_function_with_arguments.__doc__

### Real life implementation of decorator:

In [ ]:
# importing libraries 
import time 
import math 
  
# decorator to calculate duration taken by any function. 
def calculate_time(func): 
      
    # added arguments inside the inner1, if function takes any arguments, can be added like this. 
    def inner1(*args, **kwargs): 
  
        # storing time before function execution 
        begin = time.time() 
          
        func(*args, **kwargs) 
  
        # storing time after function execution 
        end = time.time() 
        print("Total time taken in : ", func.__name__, (end - begin)*1000) 
  
    return inner1 
  
  
  
# this can be added to any function present, in this case to calculate a factorial 
@calculate_time
def factorial(num):
    # sleep 2 seconds because it takes very less time so that you can see the actual difference 
    time.sleep(2) 
    print(math.factorial(num)) 

# calling the function. 
factorial(10)

In [ ]:
# Here we want to calculate performance of functions:
def cal_sq(num):
    res=[]
    for i in num:
        res.append(i*i)
    return res

def cal_cu(num):
    res=[]
    for i in num:
        res.append(i*i*i)
    return res

arr = range(1,100000)
res_sq = cal_sq(arr)
res_cu = cal_sq(arr)

# Without using decorator:
import time

def cal_sq(num):
    start= time.time() #epoch time- 1970, Jan 1
    res=[]
    for i in num:
        res.append(i*i)
    end= time.time()
    print(f'calculating sq took {(end-start)*1000} millsecond')
    return res

def cal_cu(num):
    start= time.time()
    res=[]
    for i in num:
        res.append(i*i*i)
    end= time.time()
    print(f'calculating cube took {(end-start)*1000} millsecond')
    return res

arr = range(1,1000000)
res_sq = cal_sq(arr)
res_cu = cal_cu(arr)

# Using Decorator:
import time

def cal_time(func):
    def wrapper(*args, **kwargs):
        start= time.time()
        result= func(*args, **kwargs)
        end= time.time()
        print(f'{func.__name__} took {(end-start)*1000} millsecond')
        return result
    return wrapper

@cal_time  # cal_time(cal_sq)      
def cal_sq(num):
    res=[]
    for i in num:
        res.append(i*i)
    return res

@cal_time
def cal_cu(num):
    res=[]
    for i in num:
        res.append(i*i*i)
    return res

arr = range(1,1000000)
cal_sq(arr)
cal_cu(arr)

---

### Class-based decorators

A decorator only has to be **callable** and return something callable — so a class with
`__call__` works too. This is the natural choice when the decorator needs to keep state.

Use `functools.update_wrapper(self, func)` — the class-based equivalent of `@wraps`.

In [ ]:
import functools

# A class is a decorator if its instances are callable (__call__)
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)      # the class-based equivalent of @wraps
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"  call #{self.count} to {self.func.__name__}")
        return self.func(*args, **kwargs)


@CountCalls
def greet(name):
    """Say hello."""
    return f"Hello, {name}"


print(greet("Aditya"))
print(greet("Priya"))
print("\ntotal calls:", greet.count, " <- state lives on the instance")
print("metadata kept:", greet.__name__, "|", greet.__doc__)


# ---- Decorating a method: `self` just arrives inside *args ----
def log_method(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"  {type(self).__name__}.{func.__name__}{args}")
        return func(self, *args, **kwargs)
    return wrapper


class Account:
    def __init__(self, balance=0):
        self.balance = balance

    @log_method
    def deposit(self, amount):
        self.balance += amount
        return self.balance


acct = Account(100)
print("\nbalance:", acct.deposit(50))

---

### Decorators you already use

Decorators are not an exotic technique — they are everywhere in the standard library and in
every major framework. Now that you can read one, these stop being magic:

| Decorator | From | Does |
|---|---|---|
| `@property` | builtin | Makes a method accessible like an attribute |
| `@staticmethod` / `@classmethod` | builtin | Changes how a method receives its first argument |
| `@functools.cache` / `@lru_cache` | `functools` | Memoises results |
| `@functools.wraps` | `functools` | Preserves metadata (above) |
| `@dataclass` | `dataclasses` | Generates `__init__`, `__repr__`, `__eq__` |
| `@app.route("/")` | Flask | Registers a URL handler |
| `@pytest.fixture` | pytest | Provides test setup |

In [ ]:
import functools, time

# ---- @lru_cache / @cache : memoisation, already written for you ----
@functools.cache
def slow_square(n):
    time.sleep(0.01)
    return n * n

start = time.perf_counter()
[slow_square(i % 5) for i in range(20)]
print(f"20 calls, 5 distinct: {(time.perf_counter() - start) * 1000:.0f} ms")
print("cache info:", slow_square.cache_info())


# ---- @property, @staticmethod, @classmethod : decorators you already use ----
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):
        """Computed on access, but reads like an attribute."""
        return 3.14159 * self._radius ** 2

    @staticmethod
    def describe():
        return "A circle is the set of points equidistant from a centre."

    @classmethod
    def unit(cls):
        return cls(1)


circle = Circle(2)
print(f"\n@property     : {circle.area:.2f}   (no parentheses at the call site)")
print("@staticmethod :", Circle.describe()[:34], "...")
print("@classmethod  :", f"unit circle radius {Circle.unit()._radius}")


# ---- @dataclass : generates __init__, __repr__, __eq__ ----
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

print("\n@dataclass    :", Point(1, 2), "| equality:", Point(1, 2) == Point(1, 2))


# ---- Framework decorators follow exactly the same pattern ----
routes = {}

def route(path):
    """A miniature version of Flask's @app.route."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        routes[path] = wrapper
        return wrapper
    return decorator

@route("/")
def home():
    return "Welcome"

@route("/about")
def about():
    return "About us"

print("\nregistered routes:", list(routes))
print("dispatch '/about':", routes["/about"]())

---

### `functools.singledispatch` — dispatch on the argument's type

A chain of `isinstance` checks is a common shape, and it has a real problem: to add a new
type you must edit the original function. `singledispatch` inverts that — the base function
declares a fallback, and implementations **register themselves**.

> **Version note:** `singledispatch` arrived in **3.4**; registering via a **type annotation**
> (rather than `@f.register(int)`) works from **3.7**.

In [ ]:
from functools import singledispatch

# ---- The if/elif version ----
def describe_manual(value):
    if isinstance(value, int):
        return f"an integer: {value}"
    elif isinstance(value, str):
        return f"a string of {len(value)} characters"
    elif isinstance(value, list):
        return f"a list of {len(value)} items"
    else:
        return f"something else: {type(value).__name__}"


# ---- singledispatch: one function per type, registered separately ----
@singledispatch
def describe(value):
    """Fallback, used when no registered type matches."""
    return f"something else: {type(value).__name__}"

@describe.register
def _(value: int):
    return f"an integer: {value}"

@describe.register
def _(value: str):
    return f"a string of {len(value)} characters"

@describe.register
def _(value: list):
    return f"a list of {len(value)} items"


for v in [42, "hello", [1, 2, 3], 3.14, {"a": 1}]:
    a, b = describe_manual(v), describe(v)
    assert a == b, (a, b)
    print(f"  {str(v):<12} -> {b}")

print("\nregistered types:", [t.__name__ for t in describe.registry if t is not object])

print("""
Why it is better than if/elif:
  - New types can be registered from ANOTHER module, without editing describe()
  - It respects inheritance - a subclass uses its parent's implementation
  - Each case is a normal, separately testable function
""")

# Inheritance in action
class MyList(list):
    pass

print("subclass of list ->", describe(MyList([1, 2])))

### Python Decorators Summary:
Decorators dynamically alter the functionality of a function, method, or class without having to directly use subclasses or change the source code of the function being decorated. Using decorators in Python also ensures that your code is DRY(Don't Repeat Yourself). 


Decorators have several use cases such as:
- Authorization in Python frameworks such as Flask and Django
- Logging
- Measuring execution time
- Synchronization

---

## Common Mistakes & Pitfalls

1. 🔴 **Forgetting `@functools.wraps`.** Without it the decorated function loses its `__name__`, `__doc__`, annotations and signature. `help()` becomes useless, and any tooling that reads metadata (Flask routing, pytest, Sphinx) breaks.
2. **Writing `def wrapper():` with no `*args, **kwargs`.** The decorator then only works on functions taking exactly those parameters.
3. **Forgetting to `return` inside the wrapper.** The decorated function silently returns `None` — the value is computed and thrown away.
4. **Forgetting to return the wrapper** from the decorator. The name is then bound to `None`.
5. **Confusing `@deco` with `@deco()`.** A plain decorator takes the function; a decorator *factory* must be called first. Using the wrong one gives a confusing `TypeError`.
6. **Getting the stacking order backwards.** Decorators apply bottom-up, so the one nearest the `def` wraps first and its code runs innermost.
7. **Mutable state shared across all calls** in a decorator's closure — a cache that never clears is a memory leak.
8. **Decorating a method and forgetting `self`.** `*args` absorbs it, but only if you use it.

## Best Practices

- **Always** apply `@functools.wraps(func)` to your wrapper. There is no case where you want to lose the metadata.
- Always accept `*args, **kwargs` and always `return func(*args, **kwargs)`.
- Keep the decorator's own logic small — if it needs config, make a decorator factory.
- Use `functools.lru_cache` / `cache` instead of hand-writing a memoising decorator.
- Use `functools.singledispatch` instead of a chain of `isinstance` checks.
- Prefer a plain function or a context manager when the behaviour isn't genuinely cross-cutting; decorators hide control flow.
- Give the decorator a docstring saying what it adds — the reader can't see it at the call site.

## Practice Exercises

Try these before moving on.

1. Write a `@timer` decorator with `functools.wraps` and verify `__name__` survives.
2. Write `@retry(times=3, delay=0.1)` that re-runs a failing function.
3. Write `@log_calls` that prints arguments and return value, working for any signature.
4. Stack `@timer` and `@log_calls` and demonstrate that the order changes the output.
5. Rewrite a hand-written memoiser using `functools.cache`. Compare the code.
6. Write a class-based decorator that counts how many times a function was called.
7. Use `singledispatch` to write `describe()` handling `int`, `str`, `list` and a fallback.
8. Take a decorator from earlier in this notebook and add `wraps` — then run `help()` on the decorated function before and after.